# Model Improvement 

## Material property prediction with square-net layer descriptors 

**Question**: Can we use the same preprocessed materials cohort with identical grouped splits and improve model performace? 


In [11]:
from __future__ import annotations
import os
import re
import warnings
from pathlib import Path
import pickle
from typing import Iterable, Sequence
from dataclasses import dataclass, asdict, replace
from sklearn.model_selection import ParameterGrid

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"
os.environ["OPENBLAS_NUM_THREADS"] = "2"
os.environ["NUMEXPR_NUM_THREADS"] = "2"

from notebook_utils import (
    columns_matching, material_error_analysis, ordered_unique,
    plot_binned_target, prepare_targets, read_table, save_figure,
    target_audit_table,
)
from experiment_protocol import (
    evaluate_feature_sets
)

@dataclass(frozen=True)
class ExperimentConfig:
    random_state: int = 42
    requested_splits: int = 3
    requested_repeats: int = 1
    bootstrap_draws: int = 4000
    near_hull_threshold_ev_atom: float = 0.05
    metal_gap_threshold_ev: float = 1e-6
    ridge_alpha: float = 10.0
    logistic_c: float = 1.0
    onehot_min_frequency: int = 5
    grouping_col: str = "reduced_formula_derived"
    data_path_override: str | None = None
    n_jobs: int = 2
    extra_trees_n_estimators: int = 300
    random_forest_n_estimators: int = 300
    xgb_n_estimators: int = 300
    xgb_learning_rate: float = 0.05
    xgb_max_depth: int = 5


warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)


CONFIG = ExperimentConfig()
REPO_ROOT = Path.cwd()
OUTPUT_DIR = REPO_ROOT / "outputs" / "experiment_A_compare"
FIGURE_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Feature Investigation

**Question**: Are there features that add no predicitive signal or are redundant? Could we achieve a similar performance with less features?

## 1. Reload saved data file and supplementary tables

In [15]:
#Load df from previous baseline run
DATA_PATH = OUTPUT_DIR / Path("materials_experiment_A.csv")
df = pd.read_csv(DATA_PATH)


target_registry = pd.read_csv(OUTPUT_DIR / "experiment_A_target_registry.csv")

ID_COLUMN = "material_id"
FORMULA_COLUMN = "formula"
df.head(10)

,Unnamed: 0,material_id,formula,n_layers_total,n_axes,n_species,n_pass,pass_fraction_total,has_any_pass,dominant_has_pass,dominant_axis,dominant_species,dominant_plane_id,dominant_plane_center_frac,dominant_passes2_fail_reasons,dominant_mean_score,dominant_tol_ratio_any,dominant_nn_intra_min,dominant_min_adj_dist_any_atom,dominant_uv_ang_deg_mean,dominant_uv_ang_err_mean,dominant_uv_len_err_mean,dominant_cnn_in_plane_bond_angle_deg_mean,dominant_cnn_out_of_plane_tilt_angle_deg_mean,dominant_coplane_species_counts_json,dominant_coplane_n_species,dominant_coplane_major_species,dominant_coplane_major_fraction,dominant_coplane_other_species_counts_json,dominant_coplane_other_n_species,dominant_coplane_other_fraction,dominant_cnn_out_of_plane_nn_species,dominant_cnn_out_of_plane_nn_dist,dominant_cnn_out_of_plane_bonded_species_counts_json,dominant_cnn_out_of_plane_bonded_major_species,dominant_cnn_out_of_plane_bonded_major_fraction,dominant_cnn_out_of_plane_bonded_n_species,dominant_square_species_oxi_state_mean,dominant_square_species_oxi_state_std,dominant_has_out_of_plane_same_species_bond,dominant_adj_atom_plane_major_species,dominant_adj_atom_plane_major_fraction,dominant_adj_atom_plane_species_counts_json,dominant_adj_plane_plane_major_species,dominant_adj_plane_plane_major_fraction,dominant_adj_plane_plane_species_counts_json,sg_number,sg_symbol,crystal_system,formula_pretty,...,comp__atomic_mass__std,comp__atomic_mass__min,comp__atomic_mass__max,comp__atomic_mass__range,comp__electronegativity__mean,comp__electronegativity__std,comp__electronegativity__min,comp__electronegativity__max,comp__electronegativity__range,comp__row__mean,comp__row__std,comp__row__min,comp__row__max,comp__row__range,comp__group__mean,comp__group__std,comp__group__min,comp__group__max,comp__group__range,comp__atomic_radius__mean,comp__atomic_radius__std,comp__atomic_radius__min,comp__atomic_radius__max,comp__atomic_radius__range,cv_group,chemjson__dominant_coplane_species__total,chemjson__dominant_coplane_species__richness,chemjson__dominant_coplane_species__entropy,chemjson__dominant_coplane_species__concentration,chemjson__dominant_coplane_species__max_fraction,chemjson__dominant_coplane_other_species__total,chemjson__dominant_coplane_other_species__richness,chemjson__dominant_coplane_other_species__entropy,chemjson__dominant_coplane_other_species__concentration,chemjson__dominant_coplane_other_species__max_fraction,chemjson__dominant_cnn_out_of_plane_bonded_species__total,chemjson__dominant_cnn_out_of_plane_bonded_species__richness,chemjson__dominant_cnn_out_of_plane_bonded_species__entropy,chemjson__dominant_cnn_out_of_plane_bonded_species__concentration,chemjson__dominant_cnn_out_of_plane_bonded_species__max_fraction,chemjson__dominant_adj_atom_plane_species__total,chemjson__dominant_adj_atom_plane_species__richness,chemjson__dominant_adj_atom_plane_species__entropy,chemjson__dominant_adj_atom_plane_species__concentration,chemjson__dominant_adj_atom_plane_species__max_fraction,chemjson__dominant_adj_plane_plane_species__total,chemjson__dominant_adj_plane_plane_species__richness,chemjson__dominant_adj_plane_plane_species__entropy,chemjson__dominant_adj_plane_plane_species__concentration,chemjson__dominant_adj_plane_plane_species__max_fraction
0,0,mp-1103821,Cu5Sn2Te7,52.0,3.0,3.0,0.0,0.000000,0.0,0.0,b,Cu,1.0,0.265988,primary_pass_failed,1.984846e-01,1.676686,4.289915,2.558568,89.880927,4.727615e-01,8.547092e-03,NaN,35.463778,"{""Cu"": 5}",1.0,Cu,1.00,{},0.0,NaN,Te,2.558568,"{""Te"": 20}",Te,1.0,1.0,1.2,0.4,0.0,Te,1.0,"{""Te"": 4}",Sn,1.0,"{""Sn"": 2}",5.0,C2,Monoclinic,Cu5Sn2Te7,...,29.892592,63.5460,127.600000,64.054000,2.008571,0.093416,1.90,2.10,0.20,4.642857,0.479157,4.0,5.0,1.0,13.928571,2.282364,11.0,16.0,5.0,1.340000,0.110000,1.23,1.45,0.22,formula::Cu5Sn2Te7,5.0,1.0,0.000000,1.000000,1.00,0.0,0.0,0.000000,0.000,NaN,20.0,1.0,0.0,1.0,1.0,4.0,1.0,0.000000,1.000000,1.0,2.0,1.0,0.000000,1.000000,1.0
1,1,mp-571641,PrCdPd,22.0,3.0,3.0,0.0,0.00

### Import feature-sets from initial run:

In [17]:
with open(OUTPUT_DIR / "reduced_feature_sets.pkl", "rb") as f:
    REDUCED_FEATURE_SETS = pickle.load(f)

In [ ]:
scores, predictions, manifest, failures, splits = evaluate_feature_sets(
    df,
    target_registry,
    REDUCED_FEATURE_SETS,
    CONFIG,
    model_names=["extra_trees"],
)



Band gap (eV): building shared grouped splits...
  3 paired splits (1 repeats x 3 folds)
  feature set: Baseline (38 raw columns)
    model: extra_trees


In [ ]:
print(bandgap_target)

In [ ]:
from experiment_protocol import (
    LOWER_IS_BETTER, PRIMARY_METRIC, evaluate_feature_sets,
    paired_against_baseline, primary_metric_tables,
    summarize_paired, summarize_scores,
)

score_summary = summarize_scores(scores, CONFIG)
paired_fold_differences = paired_against_baseline(scores)
paired_summary = summarize_paired(paired_fold_differences, CONFIG)
primary_score_summary, paired_primary = primary_metric_tables(
    score_summary, paired_summary, target_registry
)

primary_score_display = primary_score_summary.copy()
primary_score_display["display_rank"] = np.where(
    primary_score_display.metric.isin(LOWER_IS_BETTER),
    primary_score_display.mean_score,
    -primary_score_display.mean_score,
)
display(primary_score_display.sort_values(["target_name", "display_rank"]).drop(columns="display_rank"))
display(paired_primary.sort_values(["target_name", "mean_improvement"], ascending=[True, False]))

In [ ]:
# Primary metric score plots
for target, group in primary_score_summary.groupby('target', sort=False):
    group = group.set_index('feature_set').reindex(FEATURE_SET_ORDER).dropna(subset=['mean_score']).reset_index()
    if group.empty:
        continue
    task = group.task.iloc[0]
    metric = group.metric.iloc[0]
    fig, ax = plt.subplots(figsize=(8.8, 4.8))
    y = np.arange(len(group))
    lower = group.mean_score - group.ci_low
    upper = group.ci_high - group.mean_score
    ax.errorbar(group.mean_score, y, xerr=[lower, upper], fmt='o', capsize=4)
    ax.set_yticks(y, labels=group.feature_set)
    ax.invert_yaxis()
    ax.set_xlabel(f'{metric} (95% repeat-block bootstrap interval)')
    direction = 'lower is better' if metric in LOWER_IS_BETTER else 'higher is better'
    ax.set_title(f'{group.target_name.iloc[0]}: {metric} ({direction})')
    ax.grid(axis='x')
    save_figure(fig, f'06_score_{target}.png')
    plt.show()

# Paired fold improvement plots
for target, group in paired_fold_differences.groupby('target', sort=False):
    task = group.task.iloc[0]
    metric = PRIMARY_METRIC[task]
    plot_data = group[group.metric == metric].copy()
    if plot_data.empty and task == 'classification':
        metric = 'balanced_accuracy'
        plot_data = group[group.metric == metric].copy()
    if plot_data.empty:
        continue
    candidates = [name for name in FEATURE_SET_ORDER if name != 'Baseline' and name in set(plot_data.feature_set)]
    fig, ax = plt.subplots(figsize=(9.0, 5.2))
    rng = np.random.default_rng(CONFIG.random_state)
    for position, name in enumerate(candidates):
        values = plot_data.loc[plot_data.feature_set == name, 'improvement'].dropna().to_numpy()
        jitter = rng.normal(0, 0.045, size=len(values))
        ax.scatter(values, np.full(len(values), position) + jitter, s=28, alpha=0.65)
        if len(values):
            ax.plot([np.median(values)], [position], marker='D', markersize=7)
    ax.axvline(0, color='black', linewidth=1, linestyle='--')
    ax.set_yticks(range(len(candidates)), labels=candidates)
    ax.invert_yaxis()
    ax.set_xlabel(f'Paired improvement in {metric} vs Baseline (positive = better)')
    ax.set_title(f'{plot_data.target_name.iloc[0]}: fold-aligned paired differences')
    ax.grid(axis='x')
    save_figure(fig, f'07_paired_delta_{target}.png')
    plt.show()


In [ ]:
material_errors = material_error_analysis(predictions)
combined_material_errors = material_errors[material_errors.feature_set == 'Combined'].copy()

if not combined_material_errors.empty:
    for target, group in combined_material_errors.groupby('target', sort=False):
        display(Markdown(f"### {group.target_name.iloc[0]} - materials most helped and most hurt by Combined"))
        columns = ['material_id','formula','cv_group','y_true','mean_prediction','baseline_error','error','material_error_improvement']
        helped = group.nlargest(min(10, len(group)), 'material_error_improvement')[columns]
        hurt = group.nsmallest(min(10, len(group)), 'material_error_improvement')[columns]
        display(Markdown('**Largest error reductions**'))
        display(helped)
        display(Markdown('**Largest error increases**'))
        display(hurt)

In [ ]:

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


artifacts = {
    'fold_scores': OUTPUT_DIR / 'experiment_A_fold_scores.csv',
    'fold_predictions': OUTPUT_DIR / 'experiment_A_fold_predictions.csv',
    'split_manifest': OUTPUT_DIR / 'experiment_A_split_manifest.csv',
    'score_summary': OUTPUT_DIR / 'experiment_A_score_summary.csv',
    'paired_fold_differences': OUTPUT_DIR / 'experiment_A_paired_fold_differences.csv',
    'paired_summary': OUTPUT_DIR / 'experiment_A_paired_summary.csv',
    'material_error_analysis': OUTPUT_DIR / 'experiment_A_material_error_analysis.csv',
    'target_registry': OUTPUT_DIR / 'experiment_A_target_registry.csv',
    'family_manifest': OUTPUT_DIR / 'experiment_A_feature_family_manifest.csv',
    'feature_set_manifest': OUTPUT_DIR / 'experiment_A_feature_set_manifest.csv',
    'run_metadata': OUTPUT_DIR / 'experiment_A_run_metadata.json',
}

for frame, key in [
    (scores, 'fold_scores'),
    (predictions, 'fold_predictions'),
    (manifest, 'split_manifest'),
    (score_summary, 'score_summary'),
    (paired_fold_differences, 'paired_fold_differences'),
    (paired_summary, 'paired_summary'),
    (material_errors, 'material_error_analysis'),
    (target_registry, 'target_registry'),
    (family_manifest, 'family_manifest'),
    (set_manifest, 'feature_set_manifest'),
]:
    frame.to_csv(artifacts[key], index=False)

feature_manifest_payload = {
    'families': families,
    'feature_sets': FEATURE_SETS,
    'excluded_columns': sorted(EXCLUDED_COLUMNS),
}
with (OUTPUT_DIR / 'experiment_A_feature_manifest.json').open('w', encoding='utf-8') as handle:
    json.dump(feature_manifest_payload, handle, indent=2)

with open(OUTPUT_DIR / "feature_sets.pkl", "wb") as f:
    pickle.dump(FEATURE_SETS, f)

run_metadata = {
    'data_path': str(DATA_PATH),
    'data_sha256': sha256_file(DATA_PATH),
    'data_shape': list(df.shape),
    'id_column': ID_COLUMN,
    'formula_column': FORMULA_COLUMN,
    'group_column': CONFIG.grouping_col,
    'config': asdict(CONFIG),
}
with artifacts['run_metadata'].open('w', encoding='utf-8') as handle:
    json.dump(run_metadata, handle, indent=2)

artifact_table = pd.DataFrame([
    {'artifact': key, 'path': str(path.relative_to(REPO_ROOT)), 'exists': path.exists(), 'bytes': path.stat().st_size if path.exists() else 0}
    for key, path in artifacts.items()
] + [{
    'artifact': 'feature_manifest',
    'path': str((OUTPUT_DIR / 'experiment_A_feature_manifest.json').relative_to(REPO_ROOT)),
    'exists': (OUTPUT_DIR / 'experiment_A_feature_manifest.json').exists(),
    'bytes': (OUTPUT_DIR / 'experiment_A_feature_manifest.json').stat().st_size,
}])
display(artifact_table)

In [ ]:
target_registry_bandgap = target_registry.iloc[[0]].copy()
print(target_registry_bandgap)

In [ ]:
scores_rf, predictions_rf, manifest_rf, failures_rf, splits_rf = evaluate_feature_sets(
    df_reduced,
    target_registry_bandgap,
    REDUCED_FEATURE_SETS,
    CONFIG,
    model_names=["random_forest"],
)

In [ ]:
from experiment_protocol import (
    LOWER_IS_BETTER, PRIMARY_METRIC, evaluate_feature_sets,
    paired_against_baseline, primary_metric_tables,
    summarize_paired, summarize_scores,
)

score_summary = summarize_scores(scores_rf, CONFIG)
paired_fold_differences = paired_against_baseline(scores_rf)
paired_summary = summarize_paired(paired_fold_differences, CONFIG)
primary_score_summary, paired_primary = primary_metric_tables(
    score_summary, paired_summary, target_registry
)

primary_score_display = primary_score_summary.copy()
primary_score_display["display_rank"] = np.where(
    primary_score_display.metric.isin(LOWER_IS_BETTER),
    primary_score_display.mean_score,
    -primary_score_display.mean_score,
)
display(primary_score_display.sort_values(["target_name", "display_rank"]).drop(columns="display_rank"))
display(paired_primary.sort_values(["target_name", "mean_improvement"], ascending=[True, False]))

In [ ]:
scores_rf_full, predictions_rf_full, manifest_rf_full, failures_rf_full, splits_rf_full = evaluate_feature_sets(
    df,
    target_registry_bandgap,
    REDUCED_FEATURE_SETS,
    CONFIG,
    model_names=["random_forest"],
)